# DiffCFD — Basic Lid-Driven Cavity Simulation

This notebook demonstrates the steady-state SIMPLE solver on the classic lid-driven cavity benchmark.

The lid-driven cavity is a standard CFD validation case where the top wall moves at constant velocity
while all other walls are stationary (no-slip). The resulting recirculating flow is compared against
Ghia et al. 1982 reference data.

In [ ]:
import torch
import matplotlib.pyplot as plt
from diffcfd import NavierStokes2D

## Setup and Solve

We solve the lid-driven cavity at Re=100 on a 64×64 grid using SIMPLE with under-relaxation.

In [ ]:
solver = NavierStokes2D(
    reynolds_number=100,
    grid=(64, 64),
    device="cpu",
    max_iter=2000,
    tol=1e-5,
    alpha_u=0.7,
    alpha_p=0.3,
)

ux, uy, p = solver.solve_steady(
    sdf=None,
    lid_velocity=1.0,
    case="cavity",
)
print(f"Solve complete. Divergence: {_div(ux, uy).abs().max():.2e}")

In [ ]:
def _div(ux, uy):
    dx = dy = 1.0 / 64
    return (ux[:, 1:] - ux[:, :-1]) / dx + (uy[1:, :] - uy[:-1, :]) / dy

# Interpolate to cell centers
ux_cc = 0.5 * (ux[:, :-1] + ux[:, 1:])
uy_cc = 0.5 * (uy[:-1, :] + uy[1:, :])
speed = torch.sqrt(ux_cc**2 + uy_cc**2)

## Velocity Field

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Speed magnitude
im0 = axes[0].imshow(speed.detach().numpy(), origin='lower', cmap='viridis')
axes[0].set_title('Speed |u|')
plt.colorbar(im0, ax=axes[0])

# u-velocity
im1 = axes[1].imshow(ux_cc.detach().numpy(), origin='lower', cmap='RdBu_r')
axes[1].set_title('u-velocity')
plt.colorbar(im1, ax=axes[1])

# Pressure
im2 = axes[2].imshow(p.detach().numpy(), origin='lower', cmap='RdBu_r')
axes[2].set_title('Pressure')
plt.colorbar(im2, ax=axes[2])

plt.tight_layout()
plt.show()

## Centerline Velocity Profile vs Ghia 1982

In [ ]:
import numpy as np
from scipy.interpolate import interp1d

# Load Ghia reference data
ghia = np.loadtxt('../tests/validation/ghia1982/re100_u.csv', delimiter=',', skiprows=1)
y_ghia, u_ghia = ghia[:, 0], ghia[:, 1]

# Extract centerline u-velocity
ux_center = ux[:, 32].detach().numpy()
y_sim = np.linspace(0, 1, 64)

plt.figure(figsize=(6, 5))
plt.plot(u_ghia, y_ghia, 'ko', markersize=5, label='Ghia 1982')
plt.plot(ux_center, y_sim, 'b-', label='DiffCFD (64²)')
plt.xlabel('u-velocity')
plt.ylabel('y')
plt.title('Lid-Driven Cavity Re=100 — Centerline u-velocity')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()